# Remote-Sensing Scene Classification — Controlled Experiment Notebook

**Designed for your Windows setup and RTX 3060 GPU.** This notebook lets you proceed stage by stage in Jupyter rather than launching the whole workflow immediately.

## Your dataset locations

- **AID:** `D:\SJ\UVMDATA\AID\AID`
- **PatternNet:** `D:\SJ\UVMDATA\PatternNet` *(not included in the primary experiment)*
- **UCMerced:** `D:\SJ\UVMDATA\UCMerced_LandUse\UCMerced_landuse`
- **NWPU-RESISC45:** `D:\SJ\UVMDATA\NWPU-RESISC45`

The primary experiment uses **NWPU-RESISC45, AID, and UCMerced**. PatternNet is deliberately excluded from this first run to keep the study manageable.

### Important usage notes

1. Run cells from top to bottom.
2. Start with **Stage 1 (environment and paths)**, then **Stage 2 (audit only)**.
3. Do not run the smoke test until the audit finishes successfully.
4. Do not start the full experiment until the smoke test completes successfully.
5. If training is interrupted, rerun the **resume** cell with the same output folder and settings.
6. Do not edit or delete generated split manifests/checkpoints during the study.
7. The script checks exact and perceptual near-duplicates and keeps detected groups within partitions. Perceptual hashing is a heuristic; it does not establish geographic independence.
8. The first model creation may download pretrained weights, so internet access may be needed.

This notebook embeds the experiment code below, so you do **not** need to download a separate `.py` file.


In [1]:
from pathlib import Path
import timm
import torch
import time
import traceback

# Local directory for downloaded pretrained weights
MODEL_DIR = Path(r"D:\SJ\MDPI\pretrained_models")
MODEL_DIR.mkdir(parents=True, exist_ok=True)

# Configure Hugging Face cache location before downloading
import os
os.environ["HF_HOME"] = str(MODEL_DIR / "hf_cache")

models = {
    "ResNet50": "resnet50.tv_in1k",
    "ConvNeXt-Tiny": "convnext_tiny.fb_in1k",
    "Swin-Tiny": "swin_tiny_patch4_window7_224.ms_in1k",
    "ViT-B16": "vit_base_patch16_224.augreg_in21k_ft_in1k",
}

results = []

for name, model_id in models.items():
    print(f"\n{'=' * 60}")
    print(f"Loading: {name}")
    print(f"Identifier: {model_id}")

    start = time.time()

    try:
        model = timm.create_model(
            model_id,
            pretrained=True
        )

        model.eval()

        # Verify that the model performs a forward pass
        config = timm.data.resolve_model_data_config(model)
        size = config["input_size"][-1]

        dummy_input = torch.randn(1, 3, size, size)

        with torch.no_grad():
            output = model(dummy_input)

        print("Status: SUCCESS")
        print("Input size:", size)
        print("Output shape:", tuple(output.shape))
        print(f"Time: {time.time() - start:.1f} seconds")

        results.append({
            "model": name,
            "status": "SUCCESS",
            "seconds": round(time.time() - start, 1)
        })

        del model, dummy_input, output

    except Exception as e:
        print("Status: FAILED")
        print("Error:", repr(e))

        results.append({
            "model": name,
            "status": "FAILED",
            "seconds": round(time.time() - start, 1)
        })

print("\nDOWNLOAD SUMMARY")
for item in results:
    print(item)

print("\nHF cache location:", MODEL_DIR / "hf_cache")


Loading: ResNet50
Identifier: resnet50.tv_in1k


Status: SUCCESS
Input size: 224
Output shape: (1, 1000)
Time: 1.3 seconds

Loading: ConvNeXt-Tiny
Identifier: convnext_tiny.fb_in1k
Status: SUCCESS
Input size: 224
Output shape: (1, 1000)
Time: 0.6 seconds

Loading: Swin-Tiny
Identifier: swin_tiny_patch4_window7_224.ms_in1k
Status: SUCCESS
Input size: 224
Output shape: (1, 1000)
Time: 0.6 seconds

Loading: ViT-B16
Identifier: vit_base_patch16_224.augreg_in21k_ft_in1k
Status: SUCCESS
Input size: 224
Output shape: (1, 1000)
Time: 1.2 seconds

DOWNLOAD SUMMARY
{'model': 'ResNet50', 'status': 'SUCCESS', 'seconds': 1.3}
{'model': 'ConvNeXt-Tiny', 'status': 'SUCCESS', 'seconds': 0.6}
{'model': 'Swin-Tiny', 'status': 'SUCCESS', 'seconds': 0.6}
{'model': 'ViT-B16', 'status': 'SUCCESS', 'seconds': 1.2}

HF cache location: D:\SJ\MDPI\pretrained_models\hf_cache


## Stage 1 — Set paths and verify the environment
Run this cell first. If any path is reported missing, stop and correct it before continuing.

In [2]:
from pathlib import Path
import os, sys, json, subprocess, traceback
import torch, torchvision, timm, numpy as np, pandas as pd
from PIL import Image
import imagehash
from IPython.display import display

DATA_ROOT = Path(r"D:\SJ\UVMDATA")
OUTPUT_ROOT = Path(r"D:\SJ\MDPI\rs_experiment_v2")
SMOKE_ROOT = Path(r"D:\SJ\MDPI\rs_smoke_test")
EXPECTED_PATHS = {
    "NWPU-RESISC45": DATA_ROOT / "NWPU-RESISC45",
    "AID": DATA_ROOT / "AID" / "AID",
    "UCMerced": DATA_ROOT / "UCMerced_LandUse" / "UCMerced_landuse",
    "PatternNet (optional; not in primary run)": DATA_ROOT / "PatternNet",
}

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("timm:", timm.__version__)
print("NumPy:", np.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(f"GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

print("\nDataset path checks:")
for name, path in EXPECTED_PATHS.items():
    print(f"{name}: {'FOUND' if path.is_dir() else 'NOT FOUND'} — {path}")

missing_primary = [
    name for name in ("NWPU-RESISC45", "AID", "UCMerced")
    if not EXPECTED_PATHS[name].is_dir()
]
if missing_primary:
    raise FileNotFoundError(
        "One or more primary dataset paths are missing: " + ", ".join(missing_primary)
        + ". Stop here and correct the paths before proceeding."
    )
if not torch.cuda.is_available():
    print("\nWARNING: CUDA is unavailable. Do not start full training until GPU setup is corrected.")
else:
    print("\nEnvironment and primary dataset paths are present.")


Python: 3.11.18
PyTorch: 2.14.0+cu126
Torchvision: 0.29.0+cu126
timm: 1.0.30
NumPy: 2.4.6
CUDA available: True
GPU: NVIDIA GeForce RTX 3060
GPU memory: 12.88 GB

Dataset path checks:
NWPU-RESISC45: FOUND — D:\SJ\UVMDATA\NWPU-RESISC45
AID: FOUND — D:\SJ\UVMDATA\AID\AID
UCMerced: FOUND — D:\SJ\UVMDATA\UCMerced_LandUse\UCMerced_landuse
PatternNet (optional; not in primary run): FOUND — D:\SJ\UVMDATA\PatternNet

Environment and primary dataset paths are present.


## Stage 2 — Load the experiment functions
Run this cell once after Stage 1. It defines the audit, split, training, checkpoint, and plotting functions; it does not train models.

In [3]:
# Experiment implementation (embedded; no training starts in this cell)
"""
Resumable, controlled remote-sensing scene-classification experiments.

Design:
  Main comparison: 4 pretrained models x 3 datasets x 3 seeds at 224 px.
  Resolution sensitivity: ResNet-50 and ViT-Base/16 on RESISC45 at 112/168 px
  for the same 3 seeds. The 224-px results are reused from the main comparison.
  Total planned runs: 48 (36 main + 12 additional resolution runs).

Expected dataset roots (as supplied by the user):
  DATA_ROOT/NWPU-RESISC45/<class_name>/...
  DATA_ROOT/AID/AID/<class_name>/...
  DATA_ROOT/UCMerced_LandUse/UCMerced_landuse/<class_name>/...

The locator also accepts common alternative folder names, but selects only a folder
whose immediate subdirectories match the dataset's expected class count.

Features:
  - Per-dataset, per-seed stratified train/validation/test manifests.
  - Strict expected class-count checks (RESISC45=45, UCMerced=21, AID=30).
  - Dataset and run-configuration fingerprints; changed data/settings cannot silently reuse old runs.
  - Exact byte-duplicate detection and perceptual-hash near-duplicate grouping before splitting.
  - Group-aware stratified splits keep detected near-duplicate clusters in one partition.
  - Cross-class near-duplicate candidates abort for manual review.
  - Fine-tuning of ImageNet-pretrained models with fixed ImageNet normalization.
  - Best checkpoint selected by validation macro-F1; test evaluated once per run.
  - Epoch checkpoints, run metadata, per-epoch logs, and resumable execution.
  - Compact publication figures exported as 600-dpi PNG and vector PDF.
  - Figures use the actual reported metrics; no simulated data.

Important:
  This script prevents exact duplicates and detected perceptual near-duplicate clusters from
  crossing splits. Perceptual hashing is a heuristic, not proof of geographic independence;
  the benchmarks generally lack complete geospatial provenance. Do not claim location-grouped
  validation unless suitable source/scene metadata is available and used.
"""

from __future__ import annotations

import argparse
import csv
import hashlib
import json
import os
import random
import time
import traceback
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
from PIL import Image, ImageFile

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, f1_score,
    confusion_matrix, classification_report
)
import timm
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import imagehash

ImageFile.LOAD_TRUNCATED_IMAGES = False

# --------------------------- USER CONFIGURATION ---------------------------
DATA_ROOT = Path(r"D:\SJ\UVMDATA")  # Parent directory of the supplied dataset folders
OUTPUT_ROOT = Path(r"D:\SJ\MDPI\rs_experiment_output")

DATASET_ALIASES = {
    "RESISC45": ["NWPU-RESISC45", "RESISC45", "NWPU_RESISC45", "NWPU-RESISC45-Dataset"],
    "UCMerced": ["UCMerced_LandUse/UCMerced_landuse", "UCMerced_LandUse/Images",
                 "UCMerced_Land_Use/UCMerced_landuse", "UCMerced", "UCMerced_LandUse",
                 "UCMerced_Land_Use", "UC_Merced"],
    "AID": ["AID/AID", "AID", "AerialImageDataset", "Aerial_Image_Dataset"],
}

MODEL_IDS = {
    "resnet50": "resnet50.tv_in1k",
    "convnext_tiny": "convnext_tiny.fb_in1k",
    "swin_tiny": "swin_tiny_patch4_window7_224.ms_in1k",
    "vit_base_16": "vit_base_patch16_224.augreg_in21k_ft_in1k",
}
SEEDS = [17, 42, 2026]
PRIMARY_SIZE = 224
RESOLUTION_SIZES = [112, 168, 224]
RESOLUTION_MODELS = ["resnet50", "vit_base_16"]
DATASETS = ["RESISC45", "UCMerced", "AID"]
EXPECTED_CLASSES = {"RESISC45": 45, "UCMerced": 21, "AID": 30}

# Training settings chosen to fit a 12-GB RTX 3060 with mixed precision.
EPOCHS = 30
PATIENCE = 6
BATCH_SIZE = 32
NUM_WORKERS = 4
WEIGHT_DECAY = 1e-4
LABEL_SMOOTHING = 0.05
LR_HEAD = 3e-4
LR_BACKBONE = 3e-5
MAX_GRAD_NORM = 1.0
USE_CLASS_WEIGHTING = False  # keep standard cross-entropy for a clean baseline
PIN_MEMORY = True
# -------------------------------------------------------------------------


IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)


def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    # Determinism is best-effort; some CUDA operations may still differ by platform.
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True


def seed_worker(worker_id: int) -> None:
    """Seed Python and NumPy from the PyTorch worker seed."""
    worker_seed = torch.initial_seed() % (2 ** 32)
    np.random.seed(worker_seed)
    random.seed(worker_seed)


def json_dump(obj: Any, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    with tmp.open("w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False, default=str)
    tmp.replace(path)


def sha256_file(path: Path, chunk_size: int = 1 << 20) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(chunk_size), b""):
            h.update(chunk)
    return h.hexdigest()


def locate_dataset(dataset_name: str, data_root: Path) -> Path:
    """Locate the intended class-folder root and reject ambiguous or malformed roots."""
    if not data_root.exists():
        raise FileNotFoundError(f"DATA_ROOT does not exist: {data_root}")

    candidates = [data_root / rel for rel in DATASET_ALIASES[dataset_name]]
    # Common UC Merced archive layouts. These are explicit paths, not recursive guessing.
    if dataset_name == "UCMerced":
        candidates.extend([
            data_root / "UCMerced" / "Images",
            data_root / "UCMerced_LandUse" / "Images",
            data_root / "UCMerced_Land_Use" / "Images",
        ])

    valid = []
    expected = EXPECTED_CLASSES[dataset_name]
    for candidate in candidates:
        if not candidate.is_dir():
            continue
        class_dirs = sorted(q for q in candidate.iterdir() if q.is_dir())
        # A dataset root is valid only when immediate child folders are the class folders.
        if len(class_dirs) != expected:
            continue
        has_images = all(
            any(f.is_file() and f.suffix.lower() in IMAGE_EXTS for f in q.rglob("*"))
            for q in class_dirs
        )
        if has_images:
            valid.append(candidate.resolve())

    valid = list(dict.fromkeys(valid))
    if len(valid) == 1:
        return valid[0]
    if len(valid) > 1:
        raise RuntimeError(
            f"Multiple plausible class-folder roots found for {dataset_name}: {valid}. "
            "Please keep one canonical copy or edit DATASET_ALIASES explicitly."
        )
    expected_paths = [str(data_root / rel) for rel in DATASET_ALIASES[dataset_name]]
    raise FileNotFoundError(
        f"Could not locate a valid {dataset_name} class-folder root under {data_root}. "
        f"Checked: {expected_paths}. Expected exactly {expected} immediate class folders, "
        "each containing images. Check the folder structure; do not move or rename the data yet."
    )


class BKTree:
    """BK-tree for efficient Hamming-distance queries on 64-bit perceptual hashes."""
    def __init__(self):
        self.root = None

    @staticmethod
    def distance(a: int, b: int) -> int:
        return (a ^ b).bit_count()

    def add(self, value: int, index: int):
        if self.root is None:
            self.root = [value, [index], {}]
            return
        node = self.root
        while True:
            d = self.distance(value, node[0])
            if d == 0:
                node[1].append(index)
                return
            if d not in node[2]:
                node[2][d] = [value, [index], {}]
                return
            node = node[2][d]

    def query(self, value: int, radius: int):
        found = []
        if self.root is None:
            return found
        stack = [self.root]
        while stack:
            node = stack.pop()
            d = self.distance(value, node[0])
            if d <= radius:
                found.extend((idx, d) for idx in node[1])
            low, high = d - radius, d + radius
            stack.extend(child for edge, child in node[2].items() if low <= edge <= high)
        return found


def attach_phash_groups(rows: list[dict], threshold: int = 4) -> tuple[list[dict], dict]:
    """Cluster perceptually near-identical images; flag cross-label candidates."""
    n = len(rows)
    parent = list(range(n))
    rank = [0] * n

    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    def union(a, b):
        ra, rb = find(a), find(b)
        if ra == rb:
            return
        if rank[ra] < rank[rb]:
            ra, rb = rb, ra
        parent[rb] = ra
        if rank[ra] == rank[rb]:
            rank[ra] += 1

    tree = BKTree()
    cross_label = []
    cross_label_count = 0
    near_pairs = 0
    for i, row in enumerate(rows):
        ph = int(row["phash"], 16)
        for j, distance in tree.query(ph, threshold):
            near_pairs += 1
            if rows[i]["label"] != rows[j]["label"]:
                cross_label_count += 1
                if len(cross_label) < 100:
                    cross_label.append({
                        "path_a": rows[j]["path"], "class_a": rows[j]["class_name"],
                        "path_b": row["path"], "class_b": row["class_name"],
                        "phash_hamming_distance": distance,
                    })
            union(i, j)
        tree.add(ph, i)

    if cross_label_count:
        raise RuntimeError(
            f"Found {cross_label_count} cross-class perceptual near-duplicate candidate pair(s) "
            f"(at most 100 shown). Review these before splitting; examples: {cross_label[:5]}"
        )
    root_to_group = {}
    for i, row in enumerate(rows):
        root = find(i)
        if root not in root_to_group:
            root_to_group[root] = len(root_to_group)
        row["group_id"] = root_to_group[root]
    group_sizes = pd.Series([r["group_id"] for r in rows]).value_counts()
    audit = {
        "phash_hamming_threshold": threshold,
        "near_duplicate_pairs_detected": near_pairs,
        "near_duplicate_groups_with_multiple_images": int((group_sizes > 1).sum()),
        "largest_near_duplicate_group_size": int(group_sizes.max()) if len(group_sizes) else 0,
        "group_count": int(group_sizes.size),
        "note": "Perceptual hash grouping is heuristic; it does not establish geographic independence.",
    }
    return rows, audit


def scan_and_deduplicate(dataset_root: Path) -> tuple[list[dict], list[str], dict]:
    """Build image list; remove exact byte-identical duplicates and reject label conflicts."""
    class_dirs = sorted([p for p in dataset_root.iterdir() if p.is_dir()])
    if not class_dirs:
        raise RuntimeError(f"No class folders found under {dataset_root}")
    classes = [p.name for p in class_dirs]
    class_to_idx = {name: i for i, name in enumerate(classes)}
    by_digest: dict[str, dict] = {}
    duplicate_count = 0
    conflicting = []
    unreadable = []
    for class_dir in class_dirs:
        for path in sorted(class_dir.rglob("*")):
            if not path.is_file() or path.suffix.lower() not in IMAGE_EXTS:
                continue
            try:
                # Verify that the file is a readable image before hashing/including it.
                with Image.open(path) as im:
                    im.verify()
                digest = sha256_file(path)
                with Image.open(path) as im:
                    phash_hex = str(imagehash.phash(im.convert("RGB")))
            except Exception as e:
                unreadable.append({"path": str(path), "error": str(e)[:180]})
                continue
            label = class_to_idx[class_dir.name]
            if digest in by_digest:
                if by_digest[digest]["label"] != label:
                    conflicting.append({
                        "first_path": by_digest[digest]["path"],
                        "second_path": str(path),
                        "first_class": by_digest[digest]["class_name"],
                        "second_class": class_dir.name,
                    })
                duplicate_count += 1
                continue
            by_digest[digest] = {
                "path": str(path.resolve()), "label": label, "class_name": class_dir.name,
                "sha256": digest, "phash": phash_hex
            }
    if conflicting:
        raise RuntimeError(
            f"{len(conflicting)} byte-identical image(s) have conflicting labels in "
            f"{dataset_root}. Resolve these labels before training. Examples: "
            f"{conflicting[:3]}"
        )
    rows = list(by_digest.values())
    if not rows:
        raise RuntimeError(f"No readable images found under {dataset_root}")
    rows, phash_audit = attach_phash_groups(rows, threshold=4)
    counts = pd.Series([r["class_name"] for r in rows]).value_counts().sort_index()
    if (counts < 5).any():
        raise RuntimeError(
            "At least one class has fewer than 5 unique readable images; "
            "a reliable stratified train/validation/test split is not possible. "
            f"Counts: {counts[counts < 5].to_dict()}"
        )
    audit = {
        "dataset_root": str(dataset_root.resolve()),
        "unique_images_used": len(rows),
        "exact_duplicate_files_removed": duplicate_count,
        "unreadable_files_skipped": unreadable,
        "class_counts_after_deduplication": counts.to_dict(),
        "perceptual_hash_audit": phash_audit,
        "note": "Exact duplicates removed; perceptual near-duplicate groups kept together during splitting."
    }
    return rows, classes, audit


def make_split(rows: list[dict], seed: int) -> dict[str, list[dict]]:
    """Approximate 70/15/15 split using stratified group folds; groups never cross splits."""
    labels = np.asarray([r["label"] for r in rows])
    groups = np.asarray([r["group_id"] for r in rows])
    if len(np.unique(groups)) < 10:
        raise RuntimeError("Too few perceptual groups for a 70/15/15 group-aware split.")

    # Ten outer folds: seven folds for training and three for temporary holdout.
    outer = StratifiedGroupKFold(n_splits=10, shuffle=True, random_state=seed)
    fold_id = np.full(len(rows), -1, dtype=int)
    dummy_x = np.zeros((len(rows), 1), dtype=np.uint8)
    for fold, (_, hold_idx) in enumerate(outer.split(dummy_x, labels, groups)):
        fold_id[hold_idx] = fold
    if np.any(fold_id < 0):
        raise RuntimeError("Group-aware outer split failed to assign every image to a fold.")
    train_idx = np.where(fold_id >= 3)[0]
    temp_idx = np.where(fold_id < 3)[0]

    # Split the 30% temporary holdout into approximately equal validation and test groups.
    inner = StratifiedGroupKFold(n_splits=2, shuffle=True, random_state=seed + 1009)
    temp_groups = groups[temp_idx]
    temp_labels = labels[temp_idx]
    temp_x = np.zeros((len(temp_idx), 1), dtype=np.uint8)
    val_local, test_local = next(inner.split(temp_x, temp_labels, temp_groups))
    val_idx, test_idx = temp_idx[val_local], temp_idx[test_local]

    split = {
        "train": [rows[i] for i in train_idx],
        "val": [rows[i] for i in val_idx],
        "test": [rows[i] for i in test_idx],
    }
    for name, part in split.items():
        present = {r["label"] for r in part}
        missing = sorted(set(labels.tolist()) - present)
        if missing:
            raise RuntimeError(f"Group-aware {name} split is missing class indices: {missing}")
    return split


def dataset_fingerprint(rows: list[dict], classes: list[str]) -> str:
    """Stable fingerprint of unique image contents and assigned labels."""
    payload = {
        "classes": list(classes),
        "images": sorted((r["sha256"], int(r["label"]), r["class_name"], int(r["group_id"])) for r in rows),
    }
    raw = json.dumps(payload, sort_keys=True, separators=(",", ":")).encode("utf-8")
    return hashlib.sha256(raw).hexdigest()


def split_manifest(dataset: str, seed: int, rows: list[dict], classes: list[str],
                   out_dir: Path) -> dict[str, list[dict]]:
    manifest_path = out_dir / "splits" / dataset / f"seed_{seed}.json"
    fingerprint = dataset_fingerprint(rows, classes)
    if manifest_path.exists():
        saved = json.loads(manifest_path.read_text(encoding="utf-8"))
        if saved.get("dataset_fingerprint") != fingerprint:
            raise RuntimeError(
                f"Dataset fingerprint mismatch for {manifest_path}. The dataset contents, "
                "labels, or class list changed after the split was created. Do not reuse this "
                "output directory: archive it and deliberately create a new experiment folder."
            )
        if saved.get("classes") != classes or saved.get("seed") != seed:
            raise RuntimeError(f"Class ordering or seed mismatch in {manifest_path}.")
        split = {k: saved[k] for k in ("train", "val", "test")}
    else:
        split = make_split(rows, seed)
        payload = {
            "dataset": dataset,
            "seed": seed,
            "classes": classes,
            "dataset_fingerprint": fingerprint,
            "split_ratio_target": {"train": 0.70, "val": 0.15, "test": 0.15},
            "split_counts": {k: len(v) for k, v in split.items()},
            "split_proportions": {k: len(v) / len(rows) for k, v in split.items()},
            "group_counts": {k: len({int(r["group_id"]) for r in v}) for k, v in split.items()},
            **split,
        }
        json_dump(payload, manifest_path)

    # Explicitly assert disjoint content hashes across all partitions.
    sets = {k: {r["sha256"] for r in split[k]} for k in ("train", "val", "test")}
    if (sets["train"] & sets["val"] or sets["train"] & sets["test"]
            or sets["val"] & sets["test"]):
        raise RuntimeError(f"Exact duplicate leakage detected in split manifest {manifest_path}.")
    group_sets = {k: {int(r["group_id"]) for r in split[k]} for k in ("train", "val", "test")}
    if (group_sets["train"] & group_sets["val"] or group_sets["train"] & group_sets["test"]
            or group_sets["val"] & group_sets["test"]):
        raise RuntimeError(f"Perceptual near-duplicate group leakage detected in {manifest_path}.")
    # Assert all unique images are represented exactly once in the manifest.
    all_split_hashes = sets["train"] | sets["val"] | sets["test"]
    if all_split_hashes != {r["sha256"] for r in rows}:
        raise RuntimeError(f"Split manifest does not match the audited dataset: {manifest_path}.")
    return split


class ManifestDataset(Dataset):
    def __init__(self, rows: list[dict], transform=None):
        self.rows = rows
        self.transform = transform

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, index):
        row = self.rows[index]
        with Image.open(row["path"]) as im:
            image = im.convert("RGB")
        if self.transform:
            image = self.transform(image)
        return image, int(row["label"])


def make_transforms(size: int):
    # Aerial scene orientation is not semantically fixed; horizontal/vertical flips
    # are reasonable. Avoid aggressive color changes that may distort scene cues.
    train_tf = transforms.Compose([
        transforms.RandomResizedCrop(size, scale=(0.80, 1.0), ratio=(0.90, 1.10)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomVerticalFlip(p=0.5),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ])
    eval_tf = transforms.Compose([
        transforms.Resize((size, size), antialias=True),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ])
    return train_tf, eval_tf


def create_model(model_key: str, num_classes: int, size: int, device: torch.device):
    model_id = MODEL_IDS[model_key]
    kwargs = {}
    if model_key in {"swin_tiny", "vit_base_16"}:
        kwargs["img_size"] = size
    model = timm.create_model(
        model_id, pretrained=True, num_classes=num_classes, **kwargs
    )
    # Use discriminative learning rates: smaller updates to pretrained representation,
    # larger updates to the newly initialized classifier.
    head = model.get_classifier()
    head_params = list(head.parameters()) if isinstance(head, nn.Module) else []
    head_ids = {id(p) for p in head_params}
    backbone_params = [p for p in model.parameters() if id(p) not in head_ids]
    param_groups = [
        {"params": backbone_params, "lr": LR_BACKBONE},
        {"params": head_params, "lr": LR_HEAD},
    ]
    # Some model classifiers can be represented by a single parameter or a module;
    # verify all trainable parameters are included exactly once.
    included = {id(p) for group in param_groups for p in group["params"]}
    missing = [p for p in model.parameters() if p.requires_grad and id(p) not in included]
    if missing:
        param_groups[0]["params"].extend(missing)
    model = model.to(device)
    optimizer = torch.optim.AdamW(param_groups, weight_decay=WEIGHT_DECAY)
    return model, optimizer


def get_scaler(device: torch.device):
    return torch.amp.GradScaler("cuda", enabled=(device.type == "cuda"))


def evaluate(model, loader, device, scaler, num_classes: int, return_arrays=False):
    model.eval()
    ys, preds, probs = [], [], []
    loss_sum = 0.0
    n = 0
    criterion = nn.CrossEntropyLoss()
    with torch.inference_mode():
        for x, y in loader:
            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)
            with torch.autocast(device_type=device.type, dtype=torch.float16,
                                enabled=(device.type == "cuda")):
                logits = model(x)
                loss = criterion(logits, y)
            p = torch.softmax(logits.float(), dim=1)
            pred = p.argmax(1)
            loss_sum += float(loss.item()) * y.size(0)
            n += y.size(0)
            ys.extend(y.cpu().numpy().tolist())
            preds.extend(pred.cpu().numpy().tolist())
            probs.append(p.cpu().numpy())
    y_true = np.asarray(ys)
    y_pred = np.asarray(preds)
    prob_array = np.concatenate(probs, axis=0) if probs else np.empty((0, num_classes))
    metrics = {
        "loss": loss_sum / max(n, 1),
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "weighted_f1": f1_score(y_true, y_pred, average="weighted", zero_division=0),
    }
    if return_arrays:
        return metrics, y_true, y_pred, prob_array
    return metrics



def benchmark_inference(model, dataset: Dataset, device: torch.device, size: int, max_images: int = 200) -> dict:
    """Measure single-image latency on a fixed subset; report median and p95."""
    n = min(len(dataset), max_images)
    if n == 0:
        return {"inference_ms_per_image": None, "inference_p95_ms": None}
    loader = DataLoader(
        torch.utils.data.Subset(dataset, list(range(n))), batch_size=1, shuffle=False,
        num_workers=0, pin_memory=(device.type == "cuda")
    )
    model.eval()
    timings = []
    with torch.inference_mode():
        for i, (x, _) in enumerate(loader):
            x = x.to(device, non_blocking=True)
            if i < 10:  # warm-up passes are excluded from timing
                with torch.autocast(device_type=device.type, dtype=torch.float16,
                                    enabled=(device.type == "cuda")):
                    _ = model(x)
                if device.type == "cuda":
                    torch.cuda.synchronize()
                continue
            if device.type == "cuda":
                torch.cuda.synchronize()
            t0 = time.perf_counter()
            with torch.autocast(device_type=device.type, dtype=torch.float16,
                                enabled=(device.type == "cuda")):
                _ = model(x)
            if device.type == "cuda":
                torch.cuda.synchronize()
            timings.append((time.perf_counter() - t0) * 1000.0)
    if not timings:
        return {"inference_ms_per_image": None, "inference_p95_ms": None}
    return {
        "inference_ms_per_image": float(np.mean(timings)),
        "inference_p50_ms": float(np.median(timings)),
        "inference_p95_ms": float(np.percentile(timings, 95)),
        "inference_benchmark_images": len(timings),
    }


def run_id(dataset: str, model: str, size: int, seed: int) -> str:
    return f"{dataset}__{model}__{size}px__seed{seed}"


def run_config_payload(dataset: str, model_key: str, size: int, seed: int,
                       classes: list[str], split: dict[str, list[dict]]) -> dict:
    return {
        "dataset": dataset, "model_key": model_key, "model_id": MODEL_IDS[model_key],
        "resolution": size, "seed": seed, "classes": classes,
        "split_fingerprints": {
            k: hashlib.sha256("\n".join(sorted(r["sha256"] for r in split[k])).encode("utf-8")).hexdigest()
            for k in ("train", "val", "test")
        },
        "epochs": EPOCHS, "patience": PATIENCE, "batch_size": BATCH_SIZE,
        "num_workers": NUM_WORKERS, "weight_decay": WEIGHT_DECAY,
        "label_smoothing": LABEL_SMOOTHING, "lr_head": LR_HEAD,
        "lr_backbone": LR_BACKBONE, "max_grad_norm": MAX_GRAD_NORM,
        "train_transform": "RandomResizedCrop(scale=0.80-1.0, ratio=0.90-1.10)+HFlip+VFlip",
        "eval_transform": "Resize(square)+ImageNet normalization",
        "selection_metric": "validation macro-F1",
    }


def fingerprint_json(payload: dict) -> str:
    raw = json.dumps(payload, sort_keys=True, separators=(",", ":"), default=str).encode("utf-8")
    return hashlib.sha256(raw).hexdigest()


def train_one(dataset: str, model_key: str, size: int, seed: int,
              split: dict[str, list[dict]], classes: list[str],
              out_dir: Path, device: torch.device, resume: bool = True) -> dict:
    rid = run_id(dataset, model_key, size, seed)
    run_dir = out_dir / "runs" / rid
    run_dir.mkdir(parents=True, exist_ok=True)
    result_path = run_dir / "result.json"
    config = run_config_payload(dataset, model_key, size, seed, classes, split)
    config_fingerprint = fingerprint_json(config)
    config_path = run_dir / "run_config.json"
    if config_path.exists():
        old_config = json.loads(config_path.read_text(encoding="utf-8"))
        if old_config.get("fingerprint") != config_fingerprint:
            raise RuntimeError(
                f"Configuration mismatch for {rid}. Do not mix settings in one output folder; "
                "use a new output root for the changed experiment."
            )
    else:
        json_dump({"config": config, "fingerprint": config_fingerprint}, config_path)
    if result_path.exists():
        saved_result = json.loads(result_path.read_text(encoding="utf-8"))
        if saved_result.get("config_fingerprint") != config_fingerprint:
            raise RuntimeError(
                f"Completed result {result_path} has no matching configuration fingerprint. "
                "Archive it and use a fresh output root rather than silently reusing it."
            )
        return saved_result

    seed_everything(seed)
    train_tf, eval_tf = make_transforms(size)
    generator = torch.Generator()
    generator.manual_seed(seed)
    train_ds = ManifestDataset(split["train"], train_tf)
    val_ds = ManifestDataset(split["val"], eval_tf)
    test_ds = ManifestDataset(split["test"], eval_tf)
    loader_kwargs = {
        "num_workers": NUM_WORKERS,
        "pin_memory": PIN_MEMORY and device.type == "cuda",
        # Recreate workers each epoch so epoch-boundary resume can restore RNG seeds.
        "persistent_workers": False,
        "worker_init_fn": seed_worker,
    }
    train_loader = DataLoader(
        train_ds, batch_size=BATCH_SIZE, shuffle=True, generator=generator,
        drop_last=False, **loader_kwargs
    )
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, **loader_kwargs)
    test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, **loader_kwargs)

    model, optimizer = create_model(model_key, len(classes), size, device)
    if device.type == "cuda":
        torch.cuda.reset_peak_memory_stats(device)
    scaler = get_scaler(device)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
    criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
    ckpt_path = run_dir / "last_checkpoint.pt"
    best_path = run_dir / "best_model.pt"
    log_path = run_dir / "epochs.csv"
    start_epoch, best_f1, best_epoch, epochs_without_improvement = 0, -1.0, 0, 0

    if not resume and not result_path.exists() and any(p.exists() for p in (ckpt_path, best_path, log_path)):
        raise RuntimeError(
            f"Partial run artifacts already exist for {rid}. To avoid mixing epochs/logs, "
            "resume normally or choose a new output root; --no-resume will not overwrite them."
        )

    if resume and ckpt_path.exists():
        state = torch.load(ckpt_path, map_location="cpu", weights_only=False)
        if state.get("config_fingerprint") != config_fingerprint:
            raise RuntimeError(
                f"Checkpoint configuration mismatch for {rid}. Use the original settings or "
                "start a new output root; refusing an unsafe resume."
            )
        model.load_state_dict(state["model"])
        optimizer.load_state_dict(state["optimizer"])
        scaler.load_state_dict(state["scaler"])
        scheduler.load_state_dict(state["scheduler"])
        start_epoch = int(state["epoch"]) + 1
        best_f1 = float(state["best_f1"])
        best_epoch = int(state["best_epoch"])
        epochs_without_improvement = int(state["epochs_without_improvement"])
        if state.get("torch_rng") is not None:
            torch.set_rng_state(state["torch_rng"])
        if device.type == "cuda" and state.get("cuda_rng") is not None:
            torch.cuda.set_rng_state_all(state["cuda_rng"])
        if state.get("python_rng") is not None:
            random.setstate(state["python_rng"])
        if state.get("numpy_rng") is not None:
            np.random.set_state(state["numpy_rng"])
        if state.get("loader_generator_rng") is not None:
            generator.set_state(state["loader_generator_rng"])
        print(f"[RESUME] {rid}: continuing at epoch {start_epoch + 1}")

    if start_epoch >= EPOCHS:
        # A checkpoint can exist after the final epoch if interrupted before result write.
        if best_path.exists():
            best_state = torch.load(best_path, map_location="cpu", weights_only=False)
            if best_state.get("config_fingerprint") != config_fingerprint:
                raise RuntimeError(f"Best-model checkpoint configuration mismatch for {rid}.")
            model.load_state_dict(best_state["model"])
        else:
            raise RuntimeError(f"{rid}: completed epoch count but best checkpoint is missing.")
    else:
        for epoch in range(start_epoch, EPOCHS):
            t0 = time.time()
            model.train()
            running_loss, n_seen = 0.0, 0
            for x, y in train_loader:
                x = x.to(device, non_blocking=True)
                y = y.to(device, non_blocking=True)
                optimizer.zero_grad(set_to_none=True)
                with torch.autocast(device_type=device.type, dtype=torch.float16,
                                    enabled=(device.type == "cuda")):
                    logits = model(x)
                    loss = criterion(logits, y)
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
                scaler.step(optimizer)
                scaler.update()
                running_loss += float(loss.item()) * y.size(0)
                n_seen += y.size(0)

            val_metrics = evaluate(model, val_loader, device, scaler, len(classes))
            scheduler.step()
            improved = val_metrics["macro_f1"] > best_f1
            if improved:
                best_f1 = val_metrics["macro_f1"]
                best_epoch = epoch + 1
                epochs_without_improvement = 0
                torch.save({
                    "model": model.state_dict(), "epoch": epoch + 1,
                    "val_metrics": val_metrics, "classes": classes,
                    "model_key": model_key, "model_id": MODEL_IDS[model_key],
                    "dataset": dataset, "size": size, "seed": seed,
                    "config_fingerprint": config_fingerprint,
                }, best_path)
            else:
                epochs_without_improvement += 1

            elapsed = time.time() - t0
            row = {
                "run_id": rid, "epoch": epoch + 1,
                "train_loss": running_loss / max(n_seen, 1),
                **{f"val_{k}": v for k, v in val_metrics.items()},
                "lr_backbone": optimizer.param_groups[0]["lr"],
                "lr_head": optimizer.param_groups[1]["lr"],
                "epoch_seconds": elapsed, "best_val_macro_f1": best_f1,
                "best_epoch": best_epoch,
            }
            pd.DataFrame([row]).to_csv(
                log_path, mode="a", header=not log_path.exists(), index=False
            )
            checkpoint = {
                "epoch": epoch, "model": model.state_dict(),
                "optimizer": optimizer.state_dict(), "scaler": scaler.state_dict(),
                "scheduler": scheduler.state_dict(), "best_f1": best_f1,
                "best_epoch": best_epoch,
                "epochs_without_improvement": epochs_without_improvement,
                "torch_rng": torch.get_rng_state(),
                "cuda_rng": torch.cuda.get_rng_state_all() if device.type == "cuda" else None,
                "python_rng": random.getstate(),
                "numpy_rng": np.random.get_state(),
                "loader_generator_rng": generator.get_state(),
                "config_fingerprint": config_fingerprint,
            }
            tmp_ckpt = ckpt_path.with_suffix(".pt.tmp")
            torch.save(checkpoint, tmp_ckpt)
            tmp_ckpt.replace(ckpt_path)
            print(
                f"[{rid}] epoch {epoch + 1:02d}/{EPOCHS} "
                f"train_loss={row['train_loss']:.4f} "
                f"val_macroF1={val_metrics['macro_f1']:.4f} "
                f"best={best_f1:.4f} ({best_epoch}) {elapsed:.1f}s"
            )
            if epochs_without_improvement >= PATIENCE:
                print(f"[EARLY STOP] {rid}: no validation macro-F1 improvement for {PATIENCE} epochs.")
                break

        best_state = torch.load(best_path, map_location="cpu", weights_only=False)
        if best_state.get("config_fingerprint") != config_fingerprint:
            raise RuntimeError(f"Best-model checkpoint configuration mismatch for {rid}.")
        model.load_state_dict(best_state["model"])

    # Test set is touched only after checkpoint selection using validation macro-F1.
    test_metrics, y_true, y_pred, probs = evaluate(
        model, test_loader, device, scaler, len(classes), return_arrays=True
    )
    inference_metrics = benchmark_inference(model, test_ds, device, size)
    peak_gpu_memory_gb = (float(torch.cuda.max_memory_allocated(device) / 1e9)
                          if device.type == "cuda" else None)
    cm = confusion_matrix(y_true, y_pred, labels=np.arange(len(classes)))
    np.save(run_dir / "test_confusion_matrix.npy", cm)
    report = classification_report(
        y_true, y_pred, labels=np.arange(len(classes)), target_names=classes,
        output_dict=True, zero_division=0
    )
    pd.DataFrame(report).transpose().to_csv(run_dir / "test_classification_report.csv")
    np.savez_compressed(
        run_dir / "test_predictions.npz",
        y_true=y_true, y_pred=y_pred, probabilities=probs,
        class_names=np.asarray(classes)
    )
    result = {
        "run_id": rid, "dataset": dataset, "model": model_key,
        "model_id": MODEL_IDS[model_key], "resolution": size, "seed": seed,
        "config_fingerprint": config_fingerprint,
        "n_train": len(split["train"]), "n_val": len(split["val"]),
        "n_test": len(split["test"]), "best_epoch": int(best_state["epoch"]),
        "best_val_macro_f1": float(best_state["val_metrics"]["macro_f1"]),
        **{f"test_{k}": v for k, v in test_metrics.items()},
        **inference_metrics,
        "peak_gpu_memory_gb": peak_gpu_memory_gb,
        "status": "completed",
    }
    json_dump(result, result_path)
    # Keep best checkpoint and epoch log for reproducibility; remove large last-state checkpoint.
    if ckpt_path.exists():
        ckpt_path.unlink()
    return result


def planned_runs():
    runs = []
    # Main model comparison at 224 px on all three datasets.
    for ds in DATASETS:
        for model in MODEL_IDS:
            for seed in SEEDS:
                runs.append((ds, model, PRIMARY_SIZE, seed))
    # Controlled resolution sensitivity on the largest benchmark, RESISC45.
    # 224-px runs above are reused; only 112 and 168 px are additional runs.
    for model in RESOLUTION_MODELS:
        for size in [s for s in RESOLUTION_SIZES if s != PRIMARY_SIZE]:
            for seed in SEEDS:
                runs.append(("RESISC45", model, size, seed))
    return runs


def export_figures(results_df: pd.DataFrame, out_dir: Path):
    if results_df.empty:
        print("No completed results; figures not generated.")
        return
    fig_dir = out_dir / "figures"
    fig_dir.mkdir(parents=True, exist_ok=True)
    plt.rcParams.update({
        "font.family": "DejaVu Sans",
        "font.size": 8,
        "axes.labelsize": 8,
        "axes.titlesize": 9,
        "legend.fontsize": 7,
        "xtick.labelsize": 7,
        "ytick.labelsize": 7,
        "figure.dpi": 150,
        "savefig.dpi": 600,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
    })

    # Figure 1: Macro-F1 vs resolution on RESISC45, mean +/- SD across seeds.
    r = results_df[results_df["dataset"] == "RESISC45"].copy()
    if not r.empty:
        fig, ax = plt.subplots(figsize=(6.8, 3.8), constrained_layout=True)
        for model in sorted(r["model"].unique()):
            sub = r[r["model"] == model].groupby("resolution")["test_macro_f1"]
            stats = sub.agg(["mean", "std"]).reset_index()
            stats["std"] = stats["std"].fillna(0)
            ax.errorbar(stats["resolution"], stats["mean"], yerr=stats["std"],
                        marker="o", markersize=4, capsize=3, linewidth=1.3, label=model)
        ax.set_xlabel("Input image size (pixels)")
        ax.set_ylabel("Test macro-F1")
        ax.set_title("Resolution sensitivity on RESISC45")
        ax.set_xticks(sorted(r["resolution"].unique()))
        ax.grid(axis="y", linewidth=0.4, alpha=0.35)
        ax.legend(frameon=False, ncol=2)
        for ext in ("png", "pdf"):
            fig.savefig(fig_dir / f"fig1_resisc45_resolution.{ext}",
                        dpi=600, bbox_inches="tight", pad_inches=0.03)
        plt.close(fig)

    # Figure 2: Cross-dataset model comparison at the primary 224-px resolution.
    main = results_df[results_df["resolution"] == PRIMARY_SIZE]
    if not main.empty:
        summary = main.groupby(["dataset", "model"])["test_macro_f1"].agg(["mean", "std"]).reset_index()
        datasets = [d for d in DATASETS if d in summary["dataset"].unique()]
        models = [m for m in MODEL_IDS if m in summary["model"].unique()]
        x = np.arange(len(datasets))
        width = 0.8 / max(len(models), 1)
        fig, ax = plt.subplots(figsize=(7.0, 3.8), constrained_layout=True)
        for j, model in enumerate(models):
            sub = summary[summary["model"] == model].set_index("dataset").reindex(datasets)
            offset = (j - (len(models) - 1) / 2) * width
            ax.bar(x + offset, sub["mean"], width=width, label=model,
                   yerr=sub["std"].fillna(0), capsize=2, linewidth=0.4)
        ax.set_xticks(x, datasets)
        ax.set_ylabel("Test macro-F1 (mean ± SD)")
        ax.set_ylim(0, 1.05)
        ax.set_title("Model performance across datasets (224 × 224)")
        ax.grid(axis="y", linewidth=0.4, alpha=0.35)
        ax.legend(frameon=False, ncol=2)
        for ext in ("png", "pdf"):
            fig.savefig(fig_dir / f"fig2_cross_dataset_comparison.{ext}",
                        dpi=600, bbox_inches="tight", pad_inches=0.03)
        plt.close(fig)

    # Figure 3: Accuracy-efficiency trade-off; latency is measured on the target GPU.
    # This figure uses per-run inference timing if present in results.
    if "inference_ms_per_image" in results_df.columns:
        eff = results_df[(results_df["resolution"] == PRIMARY_SIZE) &
                         results_df["inference_ms_per_image"].notna()]
        if not eff.empty:
            fig, ax = plt.subplots(figsize=(6.8, 3.8), constrained_layout=True)
            for model in sorted(eff["model"].unique()):
                sub = eff[eff["model"] == model]
                ax.scatter(sub["inference_ms_per_image"], sub["test_macro_f1"], s=28, label=model)
            ax.set_xlabel("Inference latency (ms/image)")
            ax.set_ylabel("Test macro-F1")
            ax.set_title("Accuracy–latency trade-off")
            ax.grid(linewidth=0.4, alpha=0.35)
            ax.legend(frameon=False)
            for ext in ("png", "pdf"):
                fig.savefig(fig_dir / f"fig3_accuracy_latency.{ext}",
                            dpi=600, bbox_inches="tight", pad_inches=0.03)
            plt.close(fig)

    print(f"Figures exported to {fig_dir} (600-dpi PNG + vector PDF).")


def main():
    global EPOCHS, BATCH_SIZE, NUM_WORKERS
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--data-root", type=Path, default=DATA_ROOT)
    parser.add_argument("--output-root", type=Path, default=OUTPUT_ROOT)
    parser.add_argument("--datasets", nargs="+", choices=DATASETS, default=DATASETS)
    parser.add_argument("--models", nargs="+", choices=list(MODEL_IDS), default=list(MODEL_IDS))
    parser.add_argument("--seeds", nargs="+", type=int, default=SEEDS)
    parser.add_argument("--epochs", type=int, default=EPOCHS)
    parser.add_argument("--batch-size", type=int, default=BATCH_SIZE)
    parser.add_argument("--workers", type=int, default=NUM_WORKERS)
    parser.add_argument("--no-resume", action="store_true")
    parser.add_argument("--main-only", action="store_true",
                        help="Run only 224-px main comparison; skip resolution sensitivity.")
    parser.add_argument("--audit-only", action="store_true",
                        help="Audit datasets and create split manifests without training.")
    args = parser.parse_args()

    EPOCHS, BATCH_SIZE, NUM_WORKERS = args.epochs, args.batch_size, args.workers
    out_dir = args.output_root
    out_dir.mkdir(parents=True, exist_ok=True)

    if not torch.cuda.is_available():
        print("WARNING: CUDA is unavailable; training on CPU will be extremely slow.")
        device = torch.device("cpu")
    else:
        device = torch.device("cuda")
        print(f"GPU: {torch.cuda.get_device_name(0)} | "
              f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    print(f"PyTorch={torch.__version__} | timm={timm.__version__} | device={device}")
    print(f"Data root: {args.data_root.resolve()}")
    print(f"Output root: {out_dir.resolve()}")

    datasets_info = {}
    splits_cache = {}
    for ds in args.datasets:
        root = locate_dataset(ds, args.data_root)
        rows, classes, audit = scan_and_deduplicate(root)
        if len(classes) != EXPECTED_CLASSES[ds]:
            raise RuntimeError(
                f"{ds}: expected {EXPECTED_CLASSES[ds]} class folders but found {len(classes)} "
                f"under {root}. This usually means the wrong directory was selected or the "
                "dataset is not in its original class-folder layout. Stop and correct it."
            )
        datasets_info[ds] = {"root": str(root.resolve()), "classes": classes, "audit": audit}
        json_dump(datasets_info[ds], out_dir / "audit" / f"{ds}_audit.json")
        print(f"[AUDIT] {ds}: {len(rows)} unique images, {len(classes)} classes; "
              f"exact duplicates removed={audit['exact_duplicate_files_removed']}")
        for seed in args.seeds:
            split = split_manifest(ds, seed, rows, classes, out_dir)
            splits_cache[(ds, seed)] = (split, classes)
            print(f"[SPLIT] {ds} seed={seed}: " +
                  ", ".join(f"{k}={len(v)}" for k, v in split.items()))

    if args.audit_only:
        print("Audit-only mode complete; no models trained.")
        return

    run_plan = []
    for ds in args.datasets:
        for model in args.models:
            for seed in args.seeds:
                run_plan.append((ds, model, PRIMARY_SIZE, seed))
    if not args.main_only and "RESISC45" in args.datasets:
        for model in RESOLUTION_MODELS:
            if model not in args.models:
                continue
            for size in [s for s in RESOLUTION_SIZES if s != PRIMARY_SIZE]:
                for seed in args.seeds:
                    run_plan.append(("RESISC45", model, size, seed))

    results = []
    errors = []
    for i, (ds, model_key, size, seed) in enumerate(run_plan, start=1):
        print(f"\n===== RUN {i}/{len(run_plan)}: {run_id(ds, model_key, size, seed)} =====")
        split, classes = splits_cache[(ds, seed)]
        try:
            result = train_one(
                ds, model_key, size, seed, split, classes, out_dir, device,
                resume=not args.no_resume
            )
            results.append(result)
            # Update the aggregate CSV after every completed run, so interruption is safe.
            result_path = out_dir / "results.csv"
            all_results = []
            for p in (out_dir / "runs").glob("*/result.json"):
                try:
                    all_results.append(json.loads(p.read_text(encoding="utf-8")))
                except Exception:
                    pass
            if all_results:
                df = pd.DataFrame(all_results).sort_values(["dataset", "resolution", "model", "seed"])
                df.to_csv(result_path, index=False)
                export_figures(df, out_dir)
        except Exception as e:
            err = {"run_id": run_id(ds, model_key, size, seed),
                   "error": f"{type(e).__name__}: {e}",
                   "traceback": traceback.format_exc()}
            errors.append(err)
            json_dump(errors, out_dir / "errors.json")
            print(f"[ERROR] {err['run_id']}: {err['error']}")
            print("The script will continue to the next run. Fix the issue and rerun; "
                  "completed runs/checkpoints will be reused.")

    # Rebuild final summary from all completed run files.
    all_results = []
    for p in (out_dir / "runs").glob("*/result.json"):
        try:
            all_results.append(json.loads(p.read_text(encoding="utf-8")))
        except Exception:
            pass
    if all_results:
        df = pd.DataFrame(all_results).sort_values(["dataset", "resolution", "model", "seed"])
        df.to_csv(out_dir / "results.csv", index=False)
        export_figures(df, out_dir)
        summary = df.groupby(["dataset", "resolution", "model"]).agg(
            n_seeds=("seed", "nunique"),
            accuracy_mean=("test_accuracy", "mean"),
            accuracy_sd=("test_accuracy", "std"),
            macro_f1_mean=("test_macro_f1", "mean"),
            macro_f1_sd=("test_macro_f1", "std"),
            balanced_accuracy_mean=("test_balanced_accuracy", "mean"),
            weighted_f1_mean=("test_weighted_f1", "mean"),
            best_epoch_mean=("best_epoch", "mean"),
        ).reset_index()
        summary.to_csv(out_dir / "summary_mean_sd.csv", index=False)
        print("\nCompleted-results summary saved to summary_mean_sd.csv")
    if errors:
        print(f"{len(errors)} run(s) failed. See {out_dir / 'errors.json'}.")
    print(f"Output folder: {out_dir.resolve()}")


# Notebook mode: main() is invoked explicitly in later cells.


## Stage 3 — Audit datasets and create split manifests (no training)
This cell scans the three primary datasets, checks class counts and duplicates, and creates the split manifests in the new output folder. It may take a while because perceptual hashes are computed for the images.

**Expected classes:** NWPU-RESISC45 = 45, AID = 30, UCMerced = 21.


In [4]:
assert "main" in globals(), "Run Stage 2 first."
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
sys.argv = [
    "rs_scene_experiment_audited.py",
    "--data-root", str(DATA_ROOT),
    "--output-root", str(OUTPUT_ROOT),
    "--datasets", "RESISC45", "AID", "UCMerced",
    "--audit-only",
]
print("Starting audit-only run. No model training will occur.")
main()
print("\nAudit stage finished. Review the output above before continuing.")


Starting audit-only run. No model training will occur.
GPU: NVIDIA GeForce RTX 3060 | VRAM: 12.9 GB
PyTorch=2.14.0+cu126 | timm=1.0.30 | device=cuda
Data root: D:\SJ\UVMDATA
Output root: D:\SJ\MDPI\rs_experiment_output
[AUDIT] RESISC45: 31492 unique images, 45 classes; exact duplicates removed=8
[SPLIT] RESISC45 seed=17: train=22045, val=4724, test=4723
[SPLIT] RESISC45 seed=42: train=22044, val=4724, test=4724
[SPLIT] RESISC45 seed=2026: train=22046, val=4723, test=4723
[AUDIT] AID: 9960 unique images, 30 classes; exact duplicates removed=0
[SPLIT] AID seed=17: train=6972, val=1494, test=1494
[SPLIT] AID seed=42: train=6972, val=1494, test=1494
[SPLIT] AID seed=2026: train=6972, val=1494, test=1494
[AUDIT] UCMerced: 2100 unique images, 21 classes; exact duplicates removed=0
[SPLIT] UCMerced seed=17: train=1470, val=315, test=315
[SPLIT] UCMerced seed=42: train=1470, val=315, test=315
[SPLIT] UCMerced seed=2026: train=1470, val=315, test=315
Audit-only mode complete; no models trained.

## Stage 4 — Small smoke test (one model, one dataset, two epochs)
Run only if Stage 3 succeeds without dataset, duplicate-conflict, or split errors. This is a technical test only; **do not report its metrics as study results**. It uses a separate output directory so it cannot contaminate the primary experiment.


In [5]:
import time
import torch
import timm

device = torch.device("cuda")

model = timm.create_model(
    "resnet50.tv_in1k",
    pretrained=False,
    num_classes=45
).to(device)

model.train()
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)
criterion = torch.nn.CrossEntropyLoss()

# Synthetic data: no disk or image-loading overhead
x = torch.randn(16, 3, 224, 224, device=device)
y = torch.randint(0, 45, (16,), device=device)

# Warm-up
for _ in range(3):
    optimizer.zero_grad(set_to_none=True)
    loss = criterion(model(x), y)
    loss.backward()
    optimizer.step()

torch.cuda.synchronize()
start = time.time()

for _ in range(10):
    optimizer.zero_grad(set_to_none=True)
    loss = criterion(model(x), y)
    loss.backward()
    optimizer.step()

torch.cuda.synchronize()
elapsed = time.time() - start

print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"Average synthetic training step: {elapsed / 10:.3f} seconds")
print(f"GPU memory allocated: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")

GPU: NVIDIA GeForce RTX 3060
Average synthetic training step: 0.092 seconds
GPU memory allocated: 0.39 GB


In [6]:
# Safety check: require the audit artifacts for all three datasets.
required_audits = [
    OUTPUT_ROOT / "audit" / "RESISC45_audit.json",
    OUTPUT_ROOT / "audit" / "AID_audit.json",
    OUTPUT_ROOT / "audit" / "UCMerced_audit.json",
]
missing_audits = [str(p) for p in required_audits if not p.exists()]
if missing_audits:
    raise FileNotFoundError(
        "Audit artifacts are missing. Complete Stage 3 successfully first: "
        + ", ".join(missing_audits)
    )

SMOKE_ROOT.mkdir(parents=True, exist_ok=True)
sys.argv = [
    "rs_scene_experiment_audited.py",
    "--data-root", str(DATA_ROOT),
    "--output-root", str(SMOKE_ROOT),
    "--datasets", "RESISC45",
    "--models", "resnet50",
    "--seeds", "999",
    "--epochs", "1",
    "--batch-size", "16",
    "--workers", "0",
    "--main-only",
]
import time
import traceback

print("Starting smoke test.", flush=True)
print("Calling main()...", flush=True)

start_time = time.time()

try:
    main()
    print(
        f"\nSmoke test finished in {(time.time() - start_time) / 60:.2f} minutes.",
        flush=True
    )
except Exception:
    traceback.print_exc()


Starting smoke test.
Calling main()...
GPU: NVIDIA GeForce RTX 3060 | VRAM: 12.9 GB
PyTorch=2.14.0+cu126 | timm=1.0.30 | device=cuda
Data root: D:\SJ\UVMDATA
Output root: D:\SJ\MDPI\rs_smoke_test
[AUDIT] RESISC45: 31492 unique images, 45 classes; exact duplicates removed=8
[SPLIT] RESISC45 seed=999: train=22045, val=4723, test=4724

===== RUN 1/1: RESISC45__resnet50__224px__seed999 =====
[RESISC45__resnet50__224px__seed999] epoch 01/1 train_loss=1.1067 val_macroF1=0.9201 best=0.9201 (1) 132.8s
Figures exported to D:\SJ\MDPI\rs_smoke_test\figures (600-dpi PNG + vector PDF).
Figures exported to D:\SJ\MDPI\rs_smoke_test\figures (600-dpi PNG + vector PDF).

Completed-results summary saved to summary_mean_sd.csv
Output folder: D:\SJ\MDPI\rs_smoke_test

Smoke test finished in 3.81 minutes.


## Stage 5 — Run the primary experiment
Run this cell only after the audit and smoke test succeed.

Default study plan: **4 models × 3 datasets × 3 seeds at 224 px**, plus resolution-sensitivity runs for ResNet-50 and ViT-B/16 on NWPU-RESISC45 at 112 and 168 px. The script reuses the corresponding 224 px runs. This is computationally substantial; run only when ready.

The same output folder and settings must be used when resuming.


In [7]:
# Do not change OUTPUT_ROOT or experiment settings midway through a study.
sys.argv = [
    "rs_scene_experiment_audited.py",
    "--data-root", str(DATA_ROOT),
    "--output-root", str(OUTPUT_ROOT),
    "--datasets", "RESISC45", "AID", "UCMerced",
]
print("Starting the full primary experiment.")
main()


Starting the full primary experiment.
GPU: NVIDIA GeForce RTX 3060 | VRAM: 12.9 GB
PyTorch=2.14.0+cu126 | timm=1.0.30 | device=cuda
Data root: D:\SJ\UVMDATA
Output root: D:\SJ\MDPI\rs_experiment_output
[AUDIT] RESISC45: 31492 unique images, 45 classes; exact duplicates removed=8
[SPLIT] RESISC45 seed=17: train=22045, val=4724, test=4723
[SPLIT] RESISC45 seed=42: train=22044, val=4724, test=4724
[SPLIT] RESISC45 seed=2026: train=22046, val=4723, test=4723
[AUDIT] AID: 9960 unique images, 30 classes; exact duplicates removed=0
[SPLIT] AID seed=17: train=6972, val=1494, test=1494
[SPLIT] AID seed=42: train=6972, val=1494, test=1494
[SPLIT] AID seed=2026: train=6972, val=1494, test=1494
[AUDIT] UCMerced: 2100 unique images, 21 classes; exact duplicates removed=0
[SPLIT] UCMerced seed=17: train=1470, val=315, test=315
[SPLIT] UCMerced seed=42: train=1470, val=315, test=315
[SPLIT] UCMerced seed=2026: train=1470, val=315, test=315

===== RUN 1/48: RESISC45__resnet50__224px__seed17 =====
[RES

## Stage 6 — Resume after interruption
If Jupyter, Windows, or training stops unexpectedly, restart the kernel, run Stages 1 and 2, then run this cell. It uses the same output directory and default settings so completed runs/checkpoints can be recovered. Do not run Stage 5 and Stage 6 simultaneously.


In [8]:
sys.argv = [
    "rs_scene_experiment_audited.py",
    "--data-root", str(DATA_ROOT),
    "--output-root", str(OUTPUT_ROOT),
    "--datasets", "RESISC45", "AID", "UCMerced",
]
print("Resuming/rechecking the primary experiment using existing checkpoints and results.")
main()


Resuming/rechecking the primary experiment using existing checkpoints and results.
GPU: NVIDIA GeForce RTX 3060 | VRAM: 12.9 GB
PyTorch=2.14.0+cu126 | timm=1.0.30 | device=cuda
Data root: D:\SJ\UVMDATA
Output root: D:\SJ\MDPI\rs_experiment_output
[AUDIT] RESISC45: 31492 unique images, 45 classes; exact duplicates removed=8
[SPLIT] RESISC45 seed=17: train=22045, val=4724, test=4723
[SPLIT] RESISC45 seed=42: train=22044, val=4724, test=4724
[SPLIT] RESISC45 seed=2026: train=22046, val=4723, test=4723
[AUDIT] AID: 9960 unique images, 30 classes; exact duplicates removed=0
[SPLIT] AID seed=17: train=6972, val=1494, test=1494
[SPLIT] AID seed=42: train=6972, val=1494, test=1494
[SPLIT] AID seed=2026: train=6972, val=1494, test=1494
[AUDIT] UCMerced: 2100 unique images, 21 classes; exact duplicates removed=0
[SPLIT] UCMerced seed=17: train=1470, val=315, test=315
[SPLIT] UCMerced seed=42: train=1470, val=315, test=315
[SPLIT] UCMerced seed=2026: train=1470, val=315, test=315

===== RUN 1/48:

## Stage 7 — Inspect results and generated figures
Run after some or all runs have completed.

In [9]:
results_file = OUTPUT_ROOT / "results.csv"
summary_file = OUTPUT_ROOT / "summary_mean_sd.csv"
print("Output folder:", OUTPUT_ROOT)
print("Results CSV exists:", results_file.exists())
print("Summary CSV exists:", summary_file.exists())

if results_file.exists():
    results_df = pd.read_csv(results_file)
    display(results_df.head(10))
    print(f"Completed run records: {len(results_df)}")
if summary_file.exists():
    summary_df = pd.read_csv(summary_file)
    display(summary_df)

fig_dir = OUTPUT_ROOT / "figures"
if fig_dir.exists():
    print("\nFigure files:")
    for p in sorted(fig_dir.iterdir()):
        print(" -", p.name)
else:
    print("\nFigure folder not found yet; figures are created after successful runs.")


Output folder: D:\SJ\MDPI\rs_experiment_output
Results CSV exists: True
Summary CSV exists: True


,run_id,dataset,model,model_id,resolution,seed,config_fingerprint,n_train,n_val,n_test,...,test_accuracy,test_balanced_accuracy,test_macro_f1,test_weighted_f1,inference_ms_per_image,inference_p50_ms,inference_p95_ms,inference_benchmark_images,peak_gpu_memory_gb,status
0,AID__convnext_tiny__224px__seed17,AID,convnext_tiny,convnext_tiny.fb_in1k,224,17,f48265a78452bbf8804516e8303707f70dabb9f42460a3...,6972,1494,1494,...,0.933735,0.929143,0.929739,0.932575,8.052241,7.17760,14.568780,190,1.838657,completed
1,AID__convnext_tiny__224px__seed42,AID,convnext_tiny,convnext_tiny.fb_in1k,224,42,de2096cef9b0340660d6fc94306d930ae16f1054cc72a6...,6972,1494,1494,...,0.921017,0.914469,0.916341,0.920643,8.773755,7.23290,14.687615,190,1.838657,completed
2,AID__convnext_tiny__224px__seed2026,AID,convnext_tiny,convnext_tiny.fb_in1k,224,2026,213662ae14f3523b0ed359727ccce43121ba4a6cd6ccce...,6972,1494,1494,...,0.922356,0.916378,0.915933,0.920463,8.436901,7.20605,14.863200,190,1.838657,completed
3,AID__resnet50__224px__seed17,AID,resnet50,resnet50.tv_in1k,224,17,e96c15717e82b2f2b5a1d0871d0df1b5cd42acd46be387...,6972,1494,1494,...,0.942436,0.938086,0.939111,0.941302,8.935222,7.16515,13.384345,190,1.491192,completed
4,AID__resnet50__224px__seed42,AID,resnet50,resnet50.tv_in1k,224,42,8327c2b6119107687043ba05fb7250b47be1ecc66ba6c9...,6972,1494,1494,...,0.929719,0.924686,0.926976,0.929960,7.539168,6.77520,12.970545,190,1.491192,completed
5,AID__resnet50__224px__seed2026,AID,resnet50,resnet50.tv_in1k,224,2026,160eee9338d6dc30e64cd1761ec62c4c4682fc6ea70be2...,6972,1494,1494,...,0.930388,0.925051,0.925267,0.929433,7.022048,6.77845,8.240110,190,1.491192,completed
6,AID__swin_tiny__224px__seed17,AID,swin_tiny,swin_tiny_patch4_window7_224.ms_in1k,224,17,7c25649b6b11813d6fc5b1e17a8e7b6f5a092e4a56a4c6...,6972,1494,1494,...,0.947122,0.942227,0.943892,0.946617,13.312968,11.42890,22.906330,190,1.964665,completed
7,AID__swin_tiny__224px__seed42,AID,swin_tiny,swin_tiny_patch4_window7_224.ms_in1k,224,42,3c522deb98f78cb28bb485c10b070a79939e20911f849a...,6972,1494,1494,...,0.941098,0.938502,0.939602,0.941649,13.284785,11.47525,23.067040,190,1.964665,completed
8,AID__swin_tiny__224px__seed2026,AID,swin_tiny,swin_tiny_patch4_window7_224.ms_in1k,224,2026,c18f2c182c3e6036458cb9643fbf60510b1a4dacdce2d2...,6972,1494,1494,...,0.954485,0.950481,0.951044,0.954336,13.362254,11.38565,22.947685,190,1.964665,completed
9,AID__vit_base_16__224px__seed17,AID,vit_base_16,vit_base_patch16_224.augreg_in21k_ft_in1k,224,17,5c8c5db657b1845b7e56f5e5d4a02e5b23fd0769baaf6a...,6972,1494,1494,...,0.934404,0.933432,0.933158,0.935577,6.838052,6.03350,12.329075,190,2.761408,completed


Completed run records: 48


,dataset,resolution,model,n_seeds,accuracy_mean,accuracy_sd,macro_f1_mean,macro_f1_sd,balanced_accuracy_mean,weighted_f1_mean,best_epoch_mean
0,AID,224,convnext_tiny,3,0.925703,0.006988,0.920671,0.007855,0.919997,0.924560,1.0
1,AID,224,resnet50,3,0.934181,0.007157,0.930451,0.007548,0.929274,0.933565,1.0
2,AID,224,swin_tiny,3,0.947568,0.006705,0.944846,0.005780,0.943737,0.947534,1.0
3,AID,224,vit_base_16,3,0.929719,0.016245,0.925819,0.018152,0.927397,0.929008,1.0
4,RESISC45,112,resnet50,3,0.872406,0.008175,0.871853,0.008040,0.872367,0.871898,1.0
5,RESISC45,112,vit_base_16,3,0.894989,0.006194,0.894630,0.007444,0.894938,0.894663,1.0
6,RESISC45,168,resnet50,3,0.907410,0.007754,0.907457,0.007960,0.907365,0.907493,1.0
7,RESISC45,168,vit_base_16,3,0.916444,0.014350,0.916621,0.014267,0.916403,0.916648,1.0
8,RESISC45,224,convnext_tiny,3,0.918843,0.000846,0.918725,0.000648,0.918815,0.918749,1.0
9,RESISC45,224,resnet50,3,0.916090,0.005782,0.916141,0.005986,0.916061,0.916170,1.0



Figure files:
 - fig1_resisc45_resolution.pdf
 - fig1_resisc45_resolution.png
 - fig2_cross_dataset_comparison.pdf
 - fig2_cross_dataset_comparison.png
 - fig3_accuracy_latency.pdf
 - fig3_accuracy_latency.png


## Interpretation and reporting cautions

- Choose checkpoints using validation macro-F1; test data are used for final evaluation only.
- Report mean and standard deviation across the pre-specified seeds, and include confidence intervals/statistical tests only when computed from the actual saved predictions/results.
- Do not claim geographic or source-scene independence based only on perceptual hashing.
- If the audit flags cross-class near-duplicate candidates, manually inspect them and resolve the issue before training; do not bypass the warning merely to make the pipeline run.
- Record software versions, GPU, settings, dataset source/version, and any deviations from this protocol.
